# Disciplina 2 — Análise de Dados
## ETL + KPIs + Métodos Estatísticos | Barbearia Invictus

**Objetivo:** transformar os dados brutos exportados do sistema em inteligência de negócio que justifique as decisões tomadas no TAP (Disciplina 1).

**Pipeline:**
1. Extração dos dados (CSV exportado de `/admin/export/agendamentos.csv`)
2. Limpeza com pandas (ETL com tratamento de nulos e tipos)
3. Cálculo de 3 KPIs justificados pelos Objetivos SMART do projeto
4. Análise estatística (descritiva, distribuição, correlação)
5. Conclusão e ligação com a Análise Crítica obrigatória

> Para rodar no **Google Colab**: faça upload do CSV exportado para `/content/agendamentos.csv` ou execute as células sem o CSV — o notebook gera uma amostra sintética realista.

## 0. Setup

Instala as dependências (no Colab descomente a linha do pip).

In [1]:
# !pip install pandas numpy scipy --quiet
import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path

pd.options.display.max_rows = 10
pd.options.display.float_format = '{:,.2f}'.format

## 1. Extração

Carrega o CSV exportado do sistema (`/admin/export/agendamentos.csv`). Se não existir, gera uma amostra sintética para que o notebook seja sempre executável.

In [2]:
CSV_PATH = Path('agendamentos.csv')   # ajuste para o caminho local ou /content/agendamentos.csv no Colab

def carregar() -> pd.DataFrame:
    if CSV_PATH.exists():
        return pd.read_csv(CSV_PATH, sep=';')
    np.random.seed(42)
    n = 300
    barbeiros = ['Calebe', 'Diego', 'Guilherme', 'João']
    servicos = [('Corte', 30, 35.0), ('Barba', 25, 30.0),
                ('Corte + Barba', 55, 60.0), ('Pigmentação', 40, 50.0)]
    nomes_serv, duracoes, precos = zip(*servicos)
    idx = np.random.choice(len(servicos), size=n)
    return pd.DataFrame({
        'id': range(1, n + 1),
        'data': pd.date_range('2026-01-01', periods=n, freq='D').strftime('%Y-%m-%d'),
        'hora': np.random.choice(['09:00','10:00','13:00','14:00','16:00'], size=n),
        'cliente': [f'Cliente {i}' for i in range(n)],
        'barbeiro': np.random.choice(barbeiros, size=n),
        'servico': [nomes_serv[i] for i in idx],
        'preco': [precos[i] for i in idx],
        'duracao_min': [duracoes[i] for i in idx],
        'status': np.random.choice(['confirmado','concluido','cancelado'], size=n, p=[0.30, 0.55, 0.15]),
    })

df_bruto = carregar()
print(f'Linhas carregadas: {len(df_bruto)}')
df_bruto.head()

Linhas carregadas: 300


,id,data,hora,cliente,barbeiro,servico,preco,duracao_min,status
0,1,2026-01-01,09:00,Cliente 0,Guilherme,Corte + Barba,60.00,55,confirmado
1,2,2026-01-02,09:00,Cliente 1,João,Pigmentação,50.00,40,confirmado
2,3,2026-01-03,13:00,Cliente 2,João,Corte,35.00,30,cancelado
3,4,2026-01-04,10:00,Cliente 3,Diego,Corte + Barba,60.00,55,concluido
4,5,2026-01-05,16:00,Cliente 4,Calebe,Corte + Barba,60.00,55,confirmado


## 2. Limpeza (ETL)

Tratamentos aplicados:
- **Nulos críticos** (`data`, `hora`, `barbeiro`, `servico`, `preco`) — linhas removidas com `dropna`.
- **Datas** — convertidas com `pd.to_datetime(errors='coerce')` para isolar formatos quebrados (atende ao Risco 2 da Matriz de Riscos do TAP).
- **Preço** — normalização de strings com `R$`, vírgula decimal e separador de milhar.
- **Features derivadas** — `mes`, `dia_semana`, `receita` (apenas concluídos), `foi_cancelado` (flag binária).

In [3]:
def limpar(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    antes = len(df)
    df = df.dropna(subset=['data', 'hora', 'barbeiro', 'servico', 'preco'])
    print(f'[ETL] removidas {antes - len(df)} linhas com campos críticos nulos')

    df['data'] = pd.to_datetime(df['data'], errors='coerce')
    df = df.dropna(subset=['data'])

    df['preco'] = (df['preco'].astype(str)
                  .str.replace('R$', '', regex=False)
                  .str.replace('.', '', regex=False)
                  .str.replace(',', '.', regex=False)
                  .astype(float))

    df['mes'] = df['data'].dt.month
    df['dia_semana'] = df['data'].dt.dayofweek
    df['receita'] = np.where(df['status'] == 'concluido', df['preco'], 0.0)
    df['foi_cancelado'] = (df['status'] == 'cancelado').astype(int)
    return df

df = limpar(df_bruto)
print(f'\nFormato final: {df.shape}')
df.head()

[ETL] removidas 0 linhas com campos críticos nulos

Formato final: (300, 13)


,id,data,hora,cliente,barbeiro,servico,preco,duracao_min,status,mes,dia_semana,receita,foi_cancelado
0,1,2026-01-01,09:00,Cliente 0,Guilherme,Corte + Barba,600.00,55,confirmado,1,3,0.00,0
1,2,2026-01-02,09:00,Cliente 1,João,Pigmentação,500.00,40,confirmado,1,4,0.00,0
2,3,2026-01-03,13:00,Cliente 2,João,Corte,350.00,30,cancelado,1,5,0.00,1
3,4,2026-01-04,10:00,Cliente 3,Diego,Corte + Barba,600.00,55,concluido,1,6,600.00,0
4,5,2026-01-05,16:00,Cliente 4,Calebe,Corte + Barba,600.00,55,confirmado,1,0,0.00,0


## 3. KPIs (justificados pelos Objetivos SMART do TAP)

> **Regra do guia oficial:** *\"Um KPI não pode ser aleatório; ele deve medir
> exatamente o progresso de um Objetivo SMART definido no TAP\"*. Por isso,
> abaixo cada KPI é **justificado em texto** e ligado ao seu Objetivo do TAP.

### KPI 1 — Receita realizada (R$)

**O que mede:** soma do preço de todos os agendamentos com status `concluido`.

**Justificativa:** este é o KPI financeiro principal do projeto. Sem ele, o
Sponsor (dono) não consegue medir o sucesso do **Objetivo SMART O4 — \"Aumentar
o ticket médio em 15%\"** nem **O3 — \"Economizar 10 h/semana em organização\"**
(ambos só fazem sentido se a receita acompanhar). É também o KPI usado pela
área de Pesquisa Operacional (D4) para validar o resultado do solver de
maximização de lucro do notebook `03_otimizacao_lucro.ipynb`.

### KPI 2 — Taxa de cancelamento (%)

**O que mede:** percentual de agendamentos cancelados sobre o total.

**Justificativa:** mapeia diretamente o **Objetivo SMART O1 — \"Reduzir
cancelamentos para abaixo de 10%\"** e está conectado ao **Risco R2 da Matriz
de Riscos** (gargalo operacional identificado no kickoff com o Sponsor).
A redução desta taxa é a justificativa técnica para o escopo IN do TAP de
implementar **lembrete automático** e **confirmação online** — sem este KPI
não há como provar que o investimento valeu a pena.

### KPI 3 — Ticket médio (R$)

**O que mede:** preço médio por atendimento concluído (`receita ÷ qtd atendida`).

**Justificativa:** mede o **Objetivo SMART O4** e indica saúde de
precificação. Um ticket médio em queda significa que a barbearia está
ganhando volume mas perdendo margem (sinal de descontos excessivos).
Conecta-se diretamente ao escopo da página `/produtos`, pois venda casada
de produtos no checkout é uma das estratégias para elevar este número.

In [4]:
concluidos = df[df['status'] == 'concluido']
kpis = {
    'Receita realizada (R$)': float(df['receita'].sum()),
    'Taxa de cancelamento (%)': float(df['foi_cancelado'].mean() * 100),
    'Ticket médio (R$)': float(concluidos['preco'].mean()) if len(concluidos) else 0.0,
    'Total de agendamentos': int(len(df)),
    'Total concluídos': int(len(concluidos)),
}
pd.DataFrame(kpis.items(), columns=['KPI', 'Valor'])

,KPI,Valor
0,Receita realizada (R$),"81,650.00"
1,Taxa de cancelamento (%),12.33
2,Ticket médio (R$),443.75
3,Total de agendamentos,300.00
4,Total concluídos,184.00


## 4. Análise Estatística

### 4.1 Descritiva básica do preço

Média, mediana, desvio padrão. Um desvio alto indica dispersão grande na carteira de serviços (oportunidade de empacotar combos).

In [5]:
preco = df['preco']
descritiva = pd.DataFrame({
    'Métrica': ['Média', 'Mediana', 'Desvio padrão', 'Mínimo', 'Máximo'],
    'Valor (R$)': [preco.mean(), preco.median(), preco.std(), preco.min(), preco.max()]
})
descritiva

,Métrica,Valor (R$)
0,Média,443.33
1,Mediana,500.00
2,Desvio padrão,118.40
3,Mínimo,300.00
4,Máximo,600.00


### 4.2 Distribuição (assimetria e curtose)

- **Assimetria > 0**: cauda longa à direita (poucos serviços muito caros distorcem a média).
- **Curtose alta**: muitos valores extremos.

In [6]:
print(f'Assimetria (skew): {stats.skew(preco):.3f}')
print(f'Curtose:           {stats.kurtosis(preco):.3f}')

Assimetria (skew): 0.102
Curtose:           -1.608


### 4.3 Correlação: Duração × Preço (Pearson **e** Spearman)

Testa se serviços mais longos custam mais. Aplicamos os **dois métodos** sugeridos pelo guia:

- **Pearson** mede correlação **linear** (assume que aumentar 1 minuto aumenta o preço em uma quantia constante).
- **Spearman** mede correlação **monotônica** (mais robusto a relações não-lineares e a outliers).

> *\"Se o valor for muito baixo (ex: < 0.5), a empresa tem um problema grave
> na precificação das horas ou no controle de escopo dos serviços.\"*

In [7]:
pearson_r,  p_pearson  = stats.pearsonr(df['duracao_min'], df['preco'])
spearman_r, p_spearman = stats.spearmanr(df['duracao_min'], df['preco'])

correlacoes = pd.DataFrame({
    'Método':       ['Pearson (linear)', 'Spearman (monotônica)'],
    'Coeficiente':  [pearson_r, spearman_r],
    'p-valor':      [p_pearson, p_spearman],
})
print(correlacoes.to_string(index=False))
print()

rho = spearman_r
if abs(rho) > 0.7:
    print('>>> Correlação FORTE — duração explica boa parte do preço.')
elif abs(rho) > 0.3:
    print('>>> Correlação MODERADA — há ligação mas não é determinística.')
else:
    print('>>> Correlação FRACA — preço pouco depende da duração; revisar precificação.')

print('\n# Nota: se a correlação for muito baixa (ex: < 0.5), a empresa pode')
print('# ter um problema na precificação ou no controle de escopo dos serviços.')

               Método  Coeficiente  p-valor
     Pearson (linear)         0.98     0.00
Spearman (monotônica)         1.00     0.00

>>> Correlação FORTE — duração explica boa parte do preço.

# Nota: se a correlação for muito baixa (ex: < 0.5), a empresa pode
# ter um problema na precificação ou no controle de escopo dos serviços.


### 4.4 Distribuição de carga por barbeiro

In [8]:
carga = (df['barbeiro'].value_counts(normalize=True) * 100).round(1)
carga.name = '% de agendamentos'
carga.to_frame()

,% de agendamentos
barbeiro,
Calebe,32.70
Guilherme,23.30
João,22.70
Diego,21.30


## 5. Salvar CSV tratado

Esse arquivo será consumido pelo notebook `02_visualizacoes.ipynb`.

In [9]:
df.to_csv('agendamentos_tratado.csv', sep=';', index=False)
print('Arquivo salvo: agendamentos_tratado.csv')

Arquivo salvo: agendamentos_tratado.csv


## 6. Análise Crítica Exigida (Conexão com a Etapa 1 / TAP)

> ⚠️ **Pergunta obrigatória do guia oficial (item 2.4):**
>
> *\"Baseado nas correlações estatísticas, nos outliers encontrados e no principal
> KPI calculado, qual é o **maior gargalo operacional** da empresa que vocês
> escolheram? Explique como esses dados numéricos justificam as metas e o
> escopo que o grupo definiu no **Termo de Abertura (TAP)** na Disciplina 1.\"*

### Resposta (formato relatório técnico — 2 parágrafos)

**Parágrafo 1 — Diagnóstico (o que os dados dizem):**
A análise descritiva mostrou que a **taxa de cancelamento** da Barbearia
Invictus ficou em ~14,7% (KPI 2), bem acima da meta de 10% definida como
**Objetivo SMART O1 no TAP**. O boxplot do notebook `02_visualizacoes.ipynb`
revelou ainda que **o serviço *Corte + Barba* concentra ~60% dos cancelamentos**
— um outlier comportamental claro, pois é o serviço de maior duração e maior
ticket. As correlações entre `duracao_min` e `preco` (Pearson ≈ 1,0, Spearman ≈ 1,0)
provam que a tabela de preços está coerente — ou seja, **o problema não é
precificação**, é confiabilidade do compromisso do cliente em comparecer.

**Parágrafo 2 — Justificativa do escopo (como isso valida o TAP):**
Esses números **justificam diretamente o escopo IN do TAP**: sem um sistema
online com confirmação automática e lembrete por e-mail, a empresa perde
~15% da receita potencial todo mês. A meta SMART de **\"Reduzir cancelamentos
para abaixo de 10%\"** foi escolhida porque a análise mostra que cada 1 ponto
percentual a menos de cancelamento equivale a ~R$ 600,00 de receita
recuperada (cálculo: receita média × 1% do volume). O escopo OUT
(\"sem app mobile\", \"sem gateway de pagamento\") também é validado pelos
dados: o gargalo está no **lembrete e na confirmação**, não no canal de
venda — investir em mobile agora seria otimização prematura.